# 광안리 해변 전용 YOLO 파인튜닝 (Google Colab GPU)

이 카메라에서 수집·라벨링한 데이터로 `yolo26s`를 파인튜닝합니다.

## 사전 준비 (로컬 PC)
1. `collect_finetune_frames.py` 로 프레임 수집 (며칠, 시간대/날씨 다양)
2. `finetune/prelabel.py` 로 자동 초안 라벨 생성
3. LabelImg(YOLO) 또는 Roboflow로 `finetune/dataset/labels/` 교정
4. `finetune/make_dataset.py` 로 train/val 분할 + `gwangalli_dataset.zip` 생성
5. `gwangalli_dataset.zip` 을 Google Drive `MyDrive/gwangalli/` 에 업로드

## Colab
런타임 → 런타임 유형 변경 → **GPU (T4)** 선택 후 위에서부터 실행.

In [ ]:
# 1) GPU 확인 (없으면 런타임 유형을 GPU로 변경)
!nvidia-smi

In [ ]:
# 2) 최신 ultralytics 설치 (yolo26 지원)
# 주의: !yolo / python -m ultralytics 는 Colab에서 자주 깨짐 → 학습은 Python API만 사용
!pip -q install -U ultralytics
import ultralytics
from ultralytics import YOLO
ultralytics.checks()
print('YOLO API OK')

In [ ]:
# 3) Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4) 데이터셋 압축 해제 (Drive의 zip 경로를 맞추세요)
import zipfile, os
ZIP = '/content/drive/MyDrive/gwangalli/gwangalli_dataset.zip'
DST = '/content/gwangalli'
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DST)
# data.yaml 의 path 를 절대경로로 고정
yaml_path = os.path.join(DST, 'data.yaml')
txt = open(yaml_path).read().replace('path: .', f'path: {DST}')
open(yaml_path, 'w').write(txt)
print(txt)
print('train imgs:', len(os.listdir(os.path.join(DST, 'train/images'))))
print('val imgs:', len(os.listdir(os.path.join(DST, 'val/images'))))

In [ ]:
# 5) 파인튜닝 학습
from ultralytics import YOLO
# yolo26s 가 없으면 yolov8s 로 대체 (둘 다 소형·CPU 추론 적합)
BASE = 'yolo26s.pt'
try:
    model = YOLO(BASE)
except Exception as e:
    print('yolo26s 불가, yolov8s로 대체:', e)
    BASE = 'yolov8s.pt'
    model = YOLO(BASE)

results = model.train(
    data='/content/gwangalli/data.yaml',
    epochs=100,
    imgsz=1280,        # 원거리 소형 객체 → 고해상 학습
    batch=8,
    patience=25,       # 조기 종료
    close_mosaic=10,
    hsv_v=0.5, degrees=0.0, translate=0.05, scale=0.3, fliplr=0.5,
    project='/content/runs', name='gwangalli_ft', exist_ok=True,
)

In [ ]:
# 6) 검증 지표 확인
metrics = model.val(data='/content/gwangalli/data.yaml', imgsz=1280)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print('precision:', metrics.box.mp, 'recall:', metrics.box.mr)

In [ ]:
# 7) 최고 가중치를 Drive에 저장 + 다운로드
import shutil
best = '/content/runs/gwangalli_ft/weights/best.pt'
out = '/content/drive/MyDrive/gwangalli/yolo26s_beach_ft.pt'
shutil.copy(best, out)
print('Drive 저장:', out)
from google.colab import files
files.download(best)

## 배포 (로컬 PC)

다운로드한 `best.pt` 를 다음 경로에 저장:

```
vision/models/yolo26s_beach_ft.pt
```

그리고 서버를 재시작하면 `resolve_fast_sahi_model()` / `resolve_best_model()` 이
이 파인튜닝 모델을 **자동으로 최우선 로드**합니다. (코드 이미 대응됨)

이후 `finetune/prelabel.py` 를 새 모델로 다시 돌려 더 정확한 초안 라벨을 만들고
추가 교정→재학습하면 정확도가 계단식으로 오릅니다 (능동학습 루프).